In [1]:
# -*- coding: utf-8 -*-
from pathlib import Path
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px
import gcamreader
from utils import convert_to_mt, ej_to_twh

In [2]:
# =========================
# Config
# =========================
PROJECT_PATH = Path("/data/project/tae/gcam-core")
DB_REL_PATH  = "../output"
DB_FILE      = "database_basexdb_korea_2035"
QUERY_FILE   = Path("..") / "output" / "queries" / "Main_queries.xml"

REGION       = "South Korea"
SCENARIOS    = ["Current-Policies-Med", "Enhanced-Ambition-Med"]
Q_GEN_TECH   = 9   # electricity generation by technology
YEARS_MARK   = list(range(2015, 2036, 5))  # for share markers/annotations

# Colors
custom_colors = {
    'Solar': '#FECB52',
    'Wind': 'rgb(136,204,238)',
    'Hydro': 'rgb(95, 70, 144)',
    'Nuclear': '#AB63FA',
    'Biomass': 'rgb(115, 175, 72)',
    'Biomass w/ CCS': 'rgb(153, 201, 69)',
    'Gas w/ CCS': '#DEA0FD',
    'Gas': '#FFA15A',
    'Coal w/ CCS': '#750D86',
    'Coal': '#222A2A',
    'Oil w/ CCS': '#8D5757',
    'Oil': '#7D1215',
    'Hydrogen': "#727DCD",
    'Ammonia': "rgb(231,63,116)",
    'Others': 'rgb(217,217,217)',
}

stack_order = [
    'Ammonia', 'Coal w/ CCS', 'Coal', 'Oil w/ CCS', 'Oil',
    'Gas w/ CCS', 'Hydrogen', 'Gas', 'Nuclear',
    'Biomass', 'Hydro', 'Wind', 'Solar'
]

In [3]:
# =========================
# Helpers
# =========================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def run_query(conn, q_idx):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    df = conn.runQuery(q, scenarios=SCENARIOS, regions=[REGION])
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def cat_tech(t):
    if t in ['PV', 'PV_storage']: return 'Solar'
    if t in ['wind', 'wind_offshore', 'wind_storage']: return 'Wind'
    if t in ['hydro']: return 'Hydro'
    if t in ['Gen_III', 'Gen_II_LWR']: return 'Nuclear'
    if t in ['biomass (IGCC CCS)', 'biomass (conv CCS)']: return 'Biomass w/ CCS'
    if t in ['biomass (IGCC)', 'biomass (conv)']: return 'Biomass'
    if t in ['gas (CC CCS)']: return 'Gas w/ CCS'
    if t in ['gas (CC)', 'gas (steam/CT)']: return 'Gas'
    if t in ['coal (IGCC CCS)', 'coal (conv pul CCS)']: return 'Coal w/ CCS'
    if t in ['coal (IGCC)', 'coal (conv pul)']: return 'Coal'
    if t in ['refined liquids (CC CCS)']: return 'Oil w/ CCS'
    if t in ['refined liquids (CC)', 'refined liquids (steam/CT)']: return 'Oil'
    if t in ['gas (CC H2 blend 50%)']: return 'Hydrogen'
    if t in ['coal (conv pul ammonia blend 20%)']: return 'Ammonia'
    return 'Others'

def reallocate_h2_nh3(pivot: pd.DataFrame) -> pd.DataFrame:
    """
    Split Hydrogen & Ammonia to backing fuels (50% H2 → 50% Gas; 20% NH3 → 80% Coal)
    while preserving totals.
    """
    out = pivot.copy()
    # keep originals to compute remainders
    H_orig  = out.get('Hydrogen', pd.Series(0, index=out.index))
    NH3_orig= out.get('Ammonia',  pd.Series(0, index=out.index))

    out['Hydrogen'] = H_orig * 0.5
    out['Ammonia']  = NH3_orig * 0.2

    # Add the remaining to Gas / Coal
    out['Gas']  = out.get('Gas', 0)  + H_orig * 0.5
    out['Coal'] = out.get('Coal', 0) + NH3_orig * 0.8
    return out

def calc_shares(df_long: pd.DataFrame, scenario: str) -> tuple[pd.Series, pd.Series]:
    """Return RE% and Carbon-free% (RE + Nuclear) time series for a scenario."""
    is_re = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass'])
    is_cf = df_long['genTech'].isin(['Solar', 'Wind', 'Hydro', 'Biomass', 'Nuclear'])
    tot   = df_long[df_long['scenario'] == scenario].groupby('Year')['value'].sum()
    re    = df_long[(df_long['scenario'] == scenario) & is_re].groupby('Year')['value'].sum()
    cf    = df_long[(df_long['scenario'] == scenario) & is_cf].groupby('Year')['value'].sum()
    re_share = (re / tot * 100).reindex(YEARS_MARK)
    cf_share = (cf / tot * 100).reindex(YEARS_MARK)
    return re_share, cf_share

def add_stack_bars(fig, df_side: pd.DataFrame, col: int, show_legend: bool):
    """Add stacked bars for one subplot, using global stack_order & custom_colors."""
    cats = [c for c in stack_order if c in df_side['genTech'].unique()]
    for c in cats:
        sub = df_side[df_side['genTech'] == c]
        fig.add_bar(
            name=c, x=sub['Year'], y=sub['value'],
            marker=dict(color=custom_colors.get(c)),
            showlegend=show_legend,
            row=1, col=col, secondary_y=False
        )

In [4]:
conn = connect_db()

Database scenarios: Current-Policies-Med, Enhanced-Ambition-Med, Current-Policies-High, Current-Policies-Low, Enhanced-Ambition-High, Enhanced-Ambition-Low


In [5]:
# 1) Load generation by technology
df = run_query(conn, Q_GEN_TECH)
df['genTech'] = df['technology'].map(cat_tech)
df = df[~df['genTech'].isna()].copy()

# 2) Convert EJ → TWh
df['value'] = df['value'].apply(ej_to_twh)
df['Units'] = 'TWh'

# 3) Aggregate & reallocate H2/NH3 backing fuels
dfFig = (
    df[(df['Year'] >= 2015) & (df['Year'] <= 2035)]
    .groupby(['scenario', 'Year', 'genTech'], observed=False)['value']
    .sum().reset_index()
)

pivot = dfFig.pivot_table(index=['scenario', 'Year'], columns='genTech', values='value', fill_value=0)
pivot = reallocate_h2_nh3(pivot)

result_df = (
    pivot
    .reset_index()
    .melt(id_vars=['scenario', 'Year'], var_name='genTech', value_name='value')
)

# Order & clean
result_df['genTech'] = pd.Categorical(result_df['genTech'], categories=stack_order, ordered=True)
result_df = result_df.sort_values(['scenario', 'Year', 'genTech'])

# 4) Shares
re_current, cf_current = calc_shares(result_df, SCENARIOS[0])
re_enh,    cf_enh      = calc_shares(result_df, SCENARIOS[1])

In [6]:
# Wider subplots with less horizontal space between them
fig = make_subplots(
    rows=1, cols=2,
    shared_xaxes=True, shared_yaxes=True,
    specs=[[{"secondary_y": True}, {"secondary_y": True}]],
    subplot_titles=("Current Policies", "Enhanced Ambition"),
    column_widths=[0.5, 0.5],          # Equal width, but can make [0.55, 0.45] if needed
    horizontal_spacing=0.08            # Less space between the panels
)


# Bars: left (legend shown) & right (legend hidden)
left  = result_df[result_df['scenario'] == SCENARIOS[0]]
right = result_df[result_df['scenario'] == SCENARIOS[1]]
add_stack_bars(fig, left,  col=1, show_legend=True)
add_stack_bars(fig, right, col=2, show_legend=False)

# Share markers (RE/CF) — left
fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_current, mode="markers", name="RE Share (%)",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=True
), row=1, col=1, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_current, mode="markers", name="Carbon-Free Share (%)",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=True
), row=1, col=1, secondary_y=True)

# Share markers — right
fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=re_enh, mode="markers", name="RE Share (%)",
    marker=dict(symbol='triangle-up', size=9, color="green"),
    showlegend=False
), row=1, col=2, secondary_y=True)

fig.add_trace(go.Scatter(
    x=YEARS_MARK, y=cf_enh, mode="markers", name="Carbon-Free Share (%)",
    marker=dict(symbol='triangle-up', size=9, color="blue"),
    showlegend=False
), row=1, col=2, secondary_y=True)

# Make figure bigger
fig.update_layout(
    width=850,                         # Increase total width (was 800)
    height=700,
    barmode='stack',
    plot_bgcolor='rgba(0,0,0,0)',
    legend=dict(traceorder="reversed", font=dict(size=15), x=1.02, y=1, borderwidth=0)
)
# Primary y (left only title)
fig.update_yaxes(title="Electricity Generation (TWh)", row=1, col=1)
# Secondary y (% shares): hide ticks but keep 0–100 range
fig.update_yaxes(secondary_y=True, range=[0, 100], showticklabels=False, row=1, col=2)

# Gridlines for primary y-axes only
fig.update_yaxes(
    title_text="Electricity Generation (TWh)",
    row=1, col=1, secondary_y=False,
    showgrid=True, gridcolor='lightgray'
)
fig.update_yaxes(
    title_text=None,
    row=1, col=2, secondary_y=False,
    showgrid=True, gridcolor='lightgray'
)

# Secondary y-axes (hide grid/ticks)
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    row=1, col=1, showgrid=False
)
fig.update_yaxes(
    secondary_y=True, range=[0, 100],
    showticklabels=False, title_text=None,
    row=1, col=2, showgrid=False
)

# X axes
for c in (1, 2):
    fig.update_xaxes(
        tickmode='array', tickvals=list(range(2015, 2040, 5)),
        tickangle=45, tickfont=dict(size=15), row=1, col=c
    )

# Annotate % values near markers
for i, year in enumerate(YEARS_MARK):
    if pd.notna(re_current.get(year)):
        fig.add_annotation(x=year, y=re_current.get(year)+3, text=f"{re_current.get(year):.0f}%",
                            xref="x1", yref="y2", showarrow=False, font=dict(color="green"))
    if pd.notna(cf_current.get(year)):
        fig.add_annotation(x=year, y=cf_current.get(year)+3, text=f"{cf_current.get(year):.0f}%",
                            xref="x1", yref="y2", showarrow=False, font=dict(color="blue"))
    if pd.notna(re_enh.get(year)):
        fig.add_annotation(x=year, y=re_enh.get(year)+3, text=f"{re_enh.get(year):.0f}%",
                            xref="x2", yref="y4", showarrow=False, font=dict(color="green"))
    if pd.notna(cf_enh.get(year)):
        fig.add_annotation(x=year, y=cf_enh.get(year)+3, text=f"{cf_enh.get(year):.0f}%",
                            xref="x2", yref="y4", showarrow=False, font=dict(color="blue"))

# Fonts and titles
fig.update_xaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_yaxes(title_font=dict(size=18), tickfont=dict(size=18))
fig.update_annotations(font=dict(size=21))

pio.write_image(fig, "./fig/elec_gen_mix.png", width=800, height=700, scale=2)
fig.show()